# Lab 2 · Exercise B — The on/off problem
A counting experiment with an auxiliary measurement of the background:
$n_{\rm on}=17$ events in the signal region, $n_{\rm off}=40$ in the control region, $\tau=10.53$.

$$L(s,b)=\text{Pois}(n_{\rm on}\,|\,s+b)\;\text{Pois}(n_{\rm off}\,|\,\tau b)$$

In [ ]:
import ROOT
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

## B1 · Build and fit
Build the model with two `RooPoisson` and a `RooProdPdf`, import the single observation, fit, and read $\hat s$ and $\hat b$.
Hint: the means are `RooFormulaVar("mu_on", "@0+@1", ...)` and `RooFormulaVar("mu_off", "@0*@1", ...)`.

In [ ]:
non, noff, tau_val = 17, 40, 10.53

s   = ROOT.RooRealVar("s", "signal", 10, 0, 200)
b   = ROOT.RooRealVar("b", "background", 4, 0.01, 200)
tau = ROOT.RooRealVar("tau", "tau", tau_val); tau.setConstant(True)
n_on  = ROOT.RooRealVar("n_on", "n_on", 0, 300)
n_off = ROOT.RooRealVar("n_off", "n_off", 0, 600)

mu_on  = None   # TODO: RooFormulaVar  s + b
mu_off = None   # TODO: RooFormulaVar  tau * b
p_on   = None   # TODO: RooPoisson(n_on, mu_on)
p_off  = None   # TODO: RooPoisson(n_off, mu_off)
pdf    = None   # TODO: RooProdPdf of the two

d = ROOT.RooDataSet("d", "d", ROOT.RooArgSet(n_on, n_off))
n_on.setVal(non); n_off.setVal(noff)
d.add(ROOT.RooArgSet(n_on, n_off))

nll   = None    # TODO: pdf.createNLL(d)
minim = None    # TODO: RooMinimizer(nll); set print level -1; minimize("Minuit2", "Migrad"); hesse()
print("s_hat = %.2f +- %.2f   b_hat = %.2f +- %.2f" % (s.getVal(), s.getError(), b.getVal(), b.getError()))

In [ ]:
check("model and NLL built", pdf is not None and nll is not None and minim is not None)
check("b_hat close to n_off/tau = 3.80", abs(b.getVal() - noff / tau_val) < 0.2)
check("s_hat close to n_on - b_hat = 13.2", abs(s.getVal() - (non - noff / tau_val)) < 0.3)
check("error on b close to sqrt(n_off)/tau = 0.60", abs(b.getError() - np.sqrt(noff) / tau_val) < 0.1)

## B2 · Profile interval vs MINOS
Compute the 68 % interval on $s$ in two ways: scan the **profile** NLL until it rises by 0.5, and run **MINOS** on `s`.
`scan_interval` is provided: it needs a function returning $-\ln L_{\rm profile}(s)-(-\ln L)_{\min}$.

In [ ]:
nll_min = nll.getVal()
s_hat, b_hat = s.getVal(), b.getVal()
prof = nll.createProfile(ROOT.RooArgSet(s))

def scan_interval(fn, xmax=60, n=600):
    xs = np.linspace(0.0, xmax, n)
    ys = np.array([fn(x) for x in xs])
    ok = xs[ys <= 0.5]
    return ok.min(), ok.max()

def prof_val(x):
    s.setVal(x)
    return None   # TODO: return the profile NLL at s=x minus nll_min   (prof.getVal() - nll_min)

lo_p, hi_p = scan_interval(prof_val)
print("profile : s in [%.2f, %.2f]  ->  %+.2f / %+.2f" % (lo_p, hi_p, lo_p - s_hat, hi_p - s_hat))

s.setVal(s_hat); b.setVal(b_hat)
# TODO: run MINOS on s only, then print s.getErrorLo() and s.getErrorHi()

In [ ]:
check("profile interval contains the best fit", lo_p < s_hat < hi_p)
check("MINOS agrees with the profile scan (lower)", abs(s.getErrorLo() - (lo_p - s_hat)) < 0.3)
check("MINOS agrees with the profile scan (upper)", abs(s.getErrorHi() - (hi_p - s_hat)) < 0.3)

## B3 · Slice vs profile, and a poorer control region
`onoff_interval` repeats the whole analysis for any $(n_{\rm on}, n_{\rm off}, \tau)$ and returns the profile and the slice intervals.
Run it for the top-quark numbers and for a control region ten times smaller with the same $\hat b$: $n_{\rm off}=4$, $\tau=1.053$.

In [ ]:
def onoff_interval(non, noff, tau_val, smax=60, n=600):
    """Return (s_hat, b_hat, (lo, hi) profile, (lo, hi) slice) for the on/off model."""
    s   = ROOT.RooRealVar("s", "s", max(non - noff / tau_val, 0.5), 0, 200)
    b   = ROOT.RooRealVar("b", "b", max(noff / tau_val, 0.5), 0.01, 200)
    tau = ROOT.RooRealVar("tau", "tau", tau_val); tau.setConstant(True)
    n_on  = ROOT.RooRealVar("n_on", "n_on", 0, 300)
    n_off = ROOT.RooRealVar("n_off", "n_off", 0, 600)
    mu_on  = ROOT.RooFormulaVar("mu_on", "@0+@1", ROOT.RooArgList(s, b))
    mu_off = ROOT.RooFormulaVar("mu_off", "@0*@1", ROOT.RooArgList(tau, b))
    p_on  = ROOT.RooPoisson("p_on", "p_on", n_on, mu_on)
    p_off = ROOT.RooPoisson("p_off", "p_off", n_off, mu_off)
    pdf   = ROOT.RooProdPdf("pdf", "pdf", ROOT.RooArgList(p_on, p_off))
    d = ROOT.RooDataSet("d", "d", ROOT.RooArgSet(n_on, n_off))
    n_on.setVal(non); n_off.setVal(noff)
    d.add(ROOT.RooArgSet(n_on, n_off))

    nll = pdf.createNLL(d)
    mn = ROOT.RooMinimizer(nll); mn.setPrintLevel(-1)
    mn.minimize("Minuit2", "Migrad")
    s_hat, b_hat, nll_min = s.getVal(), b.getVal(), nll.getVal()

    prof = nll.createProfile(ROOT.RooArgSet(s))
    xs = np.linspace(0.0, smax, n)
    def prof_val(x):
        s.setVal(x); return prof.getVal() - nll_min
    def slice_val(x):
        s.setVal(x); b.setVal(b_hat); return nll.getVal() - nll_min
    def interval(fn):
        ys = np.array([fn(x) for x in xs]); ok = xs[ys <= 0.5]
        return ok.min(), ok.max()
    return s_hat, b_hat, interval(prof_val), interval(slice_val)

In [ ]:
for (a, c, t) in [(17, 40, 10.53), (17, 4, 1.053)]:
    sh, bh, prof_i, slice_i = onoff_interval(a, c, t)
    print("n_on=%d n_off=%d tau=%.3f:  s_hat=%.1f b_hat=%.2f" % (a, c, t, sh, bh))
    print("   profile  [%.2f, %.2f]  width %.2f" % (prof_i[0], prof_i[1], prof_i[1] - prof_i[0]))
    print("   slice    [%.2f, %.2f]  width %.2f" % (slice_i[0], slice_i[1], slice_i[1] - slice_i[0]))

In [ ]:
sh, bh, prof_i, slice_i = onoff_interval(17, 4, 1.053)
check("with a poor control region the profile is wider than the slice",
      (prof_i[1] - prof_i[0]) > 1.05 * (slice_i[1] - slice_i[0]))

## Questions (2–3 lines each)
1. Why are the profile interval and the slice almost equal for $\tau=10.53$, but not for $\tau=1.053$?
2. Which number carries the uncertainty on the background? Where does it come from in this model?
3. In a real analysis the control region is often a simulation or a sideband fit. What changes in the model?